In [21]:

!pip install -q -U transformers datasets accelerate scikit-learn
!pip install -q "pandas==2.2.3"

In [22]:

import os
import json
import random
import numpy as np
import pandas as pd
import torch
import transformers

from pathlib import Path

from google.colab import drive
from datasets import Dataset

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    ModernBertConfig,
    ModernBertModel,
    ModernBertPreTrainedModel,
    TrainingArguments,
    Trainer
)

from torch import nn

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Training will run on CPU.")


PyTorch: 2.14.1+cpu
Transformers: 5.23.0
GPU available: False


In [23]:

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)


Random seed: 42


In [24]:

drive.mount("/content/drive")

PROJECT_PATH = Path(
    "/content/drive/MyDrive/EthanGlobal_AI_Triage_Agent"
)

DATASET_PATH = PROJECT_PATH / "data/raw/ethan_tickets_10000.csv"

MODEL_NAME = "answerdotai/ModernBERT-base"

MODEL_OUTPUT = (
    PROJECT_PATH /
    "models/ethan_multitask_modernbert_v2"
)

REPORT_OUTPUT = (
    PROJECT_PATH /
    "reports/modernbert_v2"
)

REPORT_OUTPUT.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATASET_PATH)
print("Model output:", MODEL_OUTPUT)
print("Reports:", REPORT_OUTPUT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset: /content/drive/MyDrive/EthanGlobal_AI_Triage_Agent/data/raw/ethan_tickets_10000.csv
Model output: /content/drive/MyDrive/EthanGlobal_AI_Triage_Agent/models/ethan_multitask_modernbert_v2
Reports: /content/drive/MyDrive/EthanGlobal_AI_Triage_Agent/reports/modernbert_v2


In [25]:

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATASET_PATH}"
    )

df = pd.read_csv(DATASET_PATH)

required_columns = [
    "ticket_id",
    "text",
    "resolver_group",
    "priority",
    "parent_incident_id"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Total tickets:", len(df))
print("Columns:", df.columns.tolist())

display(df.head())


Total tickets: 10000
Columns: ['ticket_id', 'text', 'resolver_group', 'priority', 'parent_incident_id', 'client_id', 'caller_name']


,ticket_id,text,resolver_group,priority,parent_incident_id,client_id,caller_name
0,TKT006253,The server reports a hardware failure involvin...,Hardware Support,P3,INC03557,Client_B,User_4770
1,TKT004685,The web server fails to serve pages to externa...,Web Hosting,P3,INC02656,Client_B,User_4558
2,TKT001732,Users receive an installation error when updat...,Software Support,P2,INC00970,Client_B,User_8927
3,TKT004743,The collaboration platform fails to load share...,Email and Collaboration,P3,INC02693,Client_C,User_1042
4,TKT004522,The security system reports an unexpected chan...,IT Security Operations,P3,INC02560,Client_A,User_7891


In [26]:

print("Missing values:")
print(df[required_columns].isnull().sum())

print("\nDuplicate ticket IDs:", df["ticket_id"].duplicated().sum())

print(
    "Duplicate descriptions:",
    df["text"].duplicated().sum()
)

print("\nResolver groups:", df["resolver_group"].nunique())
print("Priority classes:", df["priority"].nunique())

print("\nResolver distribution:")
display(df["resolver_group"].value_counts().sort_index())

print("\nPriority distribution:")
display(df["priority"].value_counts().sort_index())

if df[required_columns].isnull().any().any():
    raise ValueError(
        "Required columns contain missing values. Resolve them before training."
    )

if df["ticket_id"].duplicated().any():
    raise ValueError("Duplicate ticket IDs detected.")

if (df["text"].astype(str).str.strip() == "").any():
    raise ValueError("Empty ticket descriptions detected.")

# Normalize label and text formatting
df["text"] = df["text"].astype(str).str.strip()
df["resolver_group"] = df["resolver_group"].astype(str).str.strip()
df["priority"] = df["priority"].astype(str).str.strip()

resolver_labels = sorted(df["resolver_group"].unique().tolist())
priority_labels = sorted(df["priority"].unique().tolist())

print("\nResolver classes:", resolver_labels)
print("Priority classes:", priority_labels)

assert len(resolver_labels) == 24, "Expected 24 resolver groups."
assert len(priority_labels) == 4, "Expected 4 priority classes."
assert set(priority_labels) == {"P1", "P2", "P3", "P4"}

print("\nData quality checks passed.")


Missing values:
ticket_id             0
text                  0
resolver_group        0
priority              0
parent_incident_id    0
dtype: int64

Duplicate ticket IDs: 0
Duplicate descriptions: 48

Resolver groups: 24
Priority classes: 4

Resolver distribution:


,count
resolver_group,
Application Support,421
Backup and Recovery,431
CRM Support,406
Cloud Infrastructure,456
Cybersecurity,375
Data Engineering,454
Database Administration,420
Desktop Support,385
DevOps,425



Priority distribution:


,count
priority,
P1,500
P2,1500
P3,5000
P4,3000



Resolver classes: ['Application Support', 'Backup and Recovery', 'CRM Support', 'Cloud Infrastructure', 'Cybersecurity', 'Data Engineering', 'Database Administration', 'Desktop Support', 'DevOps', 'ERP Support', 'Email and Collaboration', 'Hardware Support', 'IT Security Operations', 'IT Service Desk', 'Identity and Access Management', 'Linux Administration', 'Monitoring and Observability', 'Network Operations', 'Software Support', 'Storage Administration', 'Telecommunications', 'Virtualization Support', 'Web Hosting', 'Windows Administration']
Priority classes: ['P1', 'P2', 'P3', 'P4']

Data quality checks passed.


In [27]:

resolver_mapping = {
    label: index
    for index, label in enumerate(resolver_labels)
}

priority_mapping = {
    label: index
    for index, label in enumerate(priority_labels)
}

df["resolver_label"] = (
    df["resolver_group"].map(resolver_mapping)
)

df["priority_label"] = (
    df["priority"].map(priority_mapping)
)

print("Resolver mapping:")
print(resolver_mapping)

print("\nPriority mapping:")
print(priority_mapping)

assert df["resolver_label"].notna().all()
assert df["priority_label"].notna().all()

df["resolver_label"] = df["resolver_label"].astype(int)
df["priority_label"] = df["priority_label"].astype(int)

print("\nLabel encoding completed.")


Resolver mapping:
{'Application Support': 0, 'Backup and Recovery': 1, 'CRM Support': 2, 'Cloud Infrastructure': 3, 'Cybersecurity': 4, 'Data Engineering': 5, 'Database Administration': 6, 'Desktop Support': 7, 'DevOps': 8, 'ERP Support': 9, 'Email and Collaboration': 10, 'Hardware Support': 11, 'IT Security Operations': 12, 'IT Service Desk': 13, 'Identity and Access Management': 14, 'Linux Administration': 15, 'Monitoring and Observability': 16, 'Network Operations': 17, 'Software Support': 18, 'Storage Administration': 19, 'Telecommunications': 20, 'Virtualization Support': 21, 'Web Hosting': 22, 'Windows Administration': 23}

Priority mapping:
{'P1': 0, 'P2': 1, 'P3': 2, 'P4': 3}

Label encoding completed.


In [28]:

# Missing incident IDs are treated as individual incidents.
groups = df["parent_incident_id"].copy()

groups = groups.where(
    groups.notna(),
    df["ticket_id"].astype(str)
).astype(str)

# First split: 90% train/validation, 10% final test
splitter_test = GroupShuffleSplit(
    n_splits=1,
    test_size=0.10,
    random_state=SEED
)

train_val_indices, test_indices = next(
    splitter_test.split(df, groups=groups)
)

train_val_df = df.iloc[train_val_indices].copy()
test_df = df.iloc[test_indices].copy()

train_val_groups = groups.iloc[train_val_indices]

# Second split: approximately 80% train, 10% validation overall
splitter_val = GroupShuffleSplit(
    n_splits=1,
    test_size=1/9,
    random_state=SEED
)

train_indices, val_indices = next(
    splitter_val.split(
        train_val_df,
        groups=train_val_groups
    )
)

train_df = train_val_df.iloc[train_indices].copy()
val_df = train_val_df.iloc[val_indices].copy()

print("Training records:", len(train_df))
print("Validation records:", len(val_df))
print("Test records:", len(test_df))

print("\nSplit percentages:")
print("Training:", round(len(train_df) / len(df) * 100, 2), "%")
print("Validation:", round(len(val_df) / len(df) * 100, 2), "%")
print("Test:", round(len(test_df) / len(df) * 100, 2), "%")


Training records: 7977
Validation records: 1002
Test records: 1021

Split percentages:
Training: 79.77 %
Validation: 10.02 %
Test: 10.21 %


In [29]:

train_groups = set(
    groups.iloc[train_df.index].tolist()
)

val_groups = set(
    groups.iloc[val_df.index].tolist()
)

test_groups = set(
    groups.iloc[test_df.index].tolist()
)

train_val_overlap = train_groups.intersection(val_groups)
train_test_overlap = train_groups.intersection(test_groups)
val_test_overlap = val_groups.intersection(test_groups)

print("Train/validation shared incidents:", len(train_val_overlap))
print("Train/test shared incidents:", len(train_test_overlap))
print("Validation/test shared incidents:", len(val_test_overlap))

assert len(train_val_overlap) == 0
assert len(train_test_overlap) == 0
assert len(val_test_overlap) == 0

print("\nIncident-level split verification passed.")

print("\nTraining priority distribution:")
display(train_df["priority"].value_counts().sort_index())

print("\nValidation priority distribution:")
display(val_df["priority"].value_counts().sort_index())

print("\nTest priority distribution:")
display(test_df["priority"].value_counts().sort_index())


Train/validation shared incidents: 0
Train/test shared incidents: 0
Validation/test shared incidents: 0

Incident-level split verification passed.

Training priority distribution:


,count
priority,
P1,370
P2,1205
P3,4001
P4,2401



Validation priority distribution:


,count
priority,
P1,57
P2,148
P3,480
P4,317



Test priority distribution:


,count
priority,
P1,73
P2,147
P3,519
P4,282


In [30]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 128

print("Tokenizer loaded:", type(tokenizer).__name__)
print("Model:", MODEL_NAME)
print("Maximum sequence length:", MAX_LENGTH)


Tokenizer loaded: TokenizersBackend
Model: answerdotai/ModernBERT-base
Maximum sequence length: 128


In [31]:

MAX_LENGTH = 128

def encode_dataframe(dataframe):

    dataset = Dataset.from_dict({
        "text": dataframe["text"].tolist(),
        "resolver_labels": dataframe["resolver_label"].tolist(),
        "priority_labels": dataframe["priority_label"].tolist()
    })

    def tokenize_batch(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH
        )

    dataset = dataset.map(
        tokenize_batch,
        batched=True
    )

    dataset = dataset.remove_columns(["text"])

    return dataset


train_dataset = encode_dataframe(train_df)
val_dataset = encode_dataframe(val_df)
test_dataset = encode_dataframe(test_df)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

print("\nDataset features:")
print(train_dataset.features)


Map:   0%|          | 0/7977 [00:00<?, ? examples/s]

Map:   0%|          | 0/1002 [00:00<?, ? examples/s]

Map:   0%|          | 0/1021 [00:00<?, ? examples/s]

Training dataset: 7977
Validation dataset: 1002
Test dataset: 1021

Dataset features:
{'resolver_labels': Value('int64'), 'priority_labels': Value('int64'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}


In [32]:

class EthanMultiTaskBERT(ModernBertPreTrainedModel):

    # Match the encoder attribute used by ModernBERT task models.
    base_model_prefix = "model"

    def __init__(self, config):
        super().__init__(config)

        # Shared pretrained ModernBERT encoder
        self.model = ModernBertModel(config)

        # Resolver group classification: 24 classes
        self.resolver_classifier = nn.Linear(
            config.hidden_size,
            config.num_resolver_labels
        )

        # Priority classification: 4 classes
        self.priority_classifier = nn.Linear(
            config.hidden_size,
            config.num_priority_labels
        )

        dropout_probability = getattr(
            config,
            "classifier_dropout",
            0.1
        )

        self.dropout = nn.Dropout(dropout_probability)

        self.resolver_loss_fn = nn.CrossEntropyLoss()
        self.priority_loss_fn = nn.CrossEntropyLoss()

        self.post_init()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        resolver_labels=None,
        priority_labels=None
    ):

        # Extract contextual representations from ModernBERT
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Use the first token representation
        pooled_output = outputs.last_hidden_state[:, 0, :]

        pooled_output = self.dropout(pooled_output)

        # Two independent classification heads
        resolver_logits = self.resolver_classifier(pooled_output)
        priority_logits = self.priority_classifier(pooled_output)

        loss = None

        if resolver_labels is not None and priority_labels is not None:

            resolver_loss = self.resolver_loss_fn(
                resolver_logits,
                resolver_labels
            )

            priority_loss = self.priority_loss_fn(
                priority_logits,
                priority_labels
            )

            # Combined multitask loss
            loss = resolver_loss + priority_loss

        if loss is not None:
            return loss, resolver_logits, priority_logits

        return resolver_logits, priority_logits


print("EthanMultiTaskBERT defined successfully.")


EthanMultiTaskBERT defined successfully.


In [33]:

config = ModernBertConfig.from_pretrained(MODEL_NAME)

config.num_resolver_labels = len(resolver_labels)
config.num_priority_labels = len(priority_labels)

config.resolver_id2label = {
    str(i): label
    for i, label in enumerate(resolver_labels)
}

config.resolver_label2id = {
    label: i
    for i, label in enumerate(resolver_labels)
}

config.priority_id2label = {
    str(i): label
    for i, label in enumerate(priority_labels)
}

config.priority_label2id = {
    label: i
    for i, label in enumerate(priority_labels)
}

# Standard Hugging Face label metadata
config.id2label = config.resolver_id2label
config.label2id = config.resolver_label2id

model = EthanMultiTaskBERT.from_pretrained(
    MODEL_NAME,
    config=config
)

print("\nModel initialization completed.")
print("Encoder class:", type(model.model).__name__)
print("Model type:", config.model_type)
print("Encoder layers:", config.num_hidden_layers)
print("Hidden size:", config.hidden_size)
print("Resolver outputs:", config.num_resolver_labels)
print("Priority outputs:", config.num_priority_labels)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("\nTotal parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")

assert config.model_type == "modernbert"
assert isinstance(model.model, ModernBertModel)

print("\nModernBERT architecture verification passed.")


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] EthanMultiTaskBERT LOAD REPORT from: answerdotai/ModernBERT-base
Key                        | Status     | 
---------------------------+------------+-
decoder.bias               | UNEXPECTED | 
head.dense.weight          | UNEXPECTED | 
head.norm.weight           | UNEXPECTED | 
resolver_classifier.weight | MISSING    | 
resolver_classifier.bias   | MISSING    | 
priority_classifier.weight | MISSING    | 
priority_classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Model initialization completed.
Encoder class: ModernBertModel
Model type: modernbert
Encoder layers: 22
Hidden size: 768
Resolver outputs: 24
Priority outputs: 4

Total parameters: 149,035,804
Trainable parameters: 149,035,804

ModernBERT architecture verification passed.


In [34]:

def compute_metrics(prediction):

    resolver_logits, priority_logits = prediction.predictions

    resolver_true, priority_true = prediction.label_ids

    resolver_pred = np.argmax(resolver_logits, axis=-1)
    priority_pred = np.argmax(priority_logits, axis=-1)

    resolver_precision, resolver_recall, resolver_f1, _ = (
        precision_recall_fscore_support(
            resolver_true,
            resolver_pred,
            average="macro",
            zero_division=0
        )
    )

    priority_precision, priority_recall, priority_f1, _ = (
        precision_recall_fscore_support(
            priority_true,
            priority_pred,
            average="macro",
            zero_division=0
        )
    )

    return {
        "resolver_accuracy": accuracy_score(
            resolver_true,
            resolver_pred
        ),
        "resolver_macro_precision": resolver_precision,
        "resolver_macro_recall": resolver_recall,
        "resolver_macro_f1": resolver_f1,

        "priority_accuracy": accuracy_score(
            priority_true,
            priority_pred
        ),
        "priority_macro_precision": priority_precision,
        "priority_macro_recall": priority_recall,
        "priority_macro_f1": priority_f1
    }


print("Evaluation metrics configured.")


Evaluation metrics configured.


In [35]:

import math

BATCH_SIZE = 16
EPOCHS = 3

total_training_steps = (
    math.ceil(len(train_dataset) / BATCH_SIZE) * EPOCHS
)

warmup_steps = int(total_training_steps * 0.1)

training_args = TrainingArguments(
    output_dir="/content/ethan_modernbert_v2_checkpoints",

    num_train_epochs=EPOCHS,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=warmup_steps,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_resolver_macro_f1",
    greater_is_better=True,

    save_total_limit=2,
    logging_strategy="steps",
    logging_steps=50,

    fp16=torch.cuda.is_available(),
    report_to="none",

    seed=SEED,
    data_seed=SEED
)

print("Training configuration ready.")
print("Warmup steps:", warmup_steps)



Training configuration ready.
Warmup steps: 149


In [36]:

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    compute_metrics=compute_metrics
)

print("Trainer initialized.")
print("Training records:", len(train_dataset))
print("Validation records:", len(val_dataset))


Trainer initialized.
Training records: 7977
Validation records: 1002


In [ ]:

training_result = trainer.train()

print("\nTraining completed.")

print("\nTraining metrics:")
for key, value in training_result.metrics.items():
    print(f"{key}: {value}")


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:

test_predictions = trainer.predict(test_dataset)

resolver_logits, priority_logits = test_predictions.predictions

resolver_true, priority_true = test_predictions.label_ids

resolver_pred = np.argmax(resolver_logits, axis=-1)
priority_pred = np.argmax(priority_logits, axis=-1)

resolver_report = classification_report(
    resolver_true,
    resolver_pred,
    labels=list(range(len(resolver_labels))),
    target_names=resolver_labels,
    output_dict=True,
    zero_division=0
)

priority_report = classification_report(
    priority_true,
    priority_pred,
    labels=list(range(len(priority_labels))),
    target_names=priority_labels,
    output_dict=True,
    zero_division=0
)

print("RESOLVER CLASSIFICATION REPORT")
display(pd.DataFrame(resolver_report).transpose().round(4))

print("\nPRIORITY CLASSIFICATION REPORT")
display(pd.DataFrame(priority_report).transpose().round(4))


In [ ]:

p1_index = priority_mapping["P1"]

p1_true = (priority_true == p1_index).astype(int)
p1_pred = (priority_pred == p1_index).astype(int)

p1_precision, p1_recall, p1_f1, p1_support = (
    precision_recall_fscore_support(
        p1_true,
        p1_pred,
        average="binary",
        zero_division=0
    )
)

print("P1 EVALUATION")
print("P1 precision:", round(p1_precision, 4))
print("P1 recall:", round(p1_recall, 4))
print("P1 F1:", round(p1_f1, 4))
print("P1 actual test tickets:", int(p1_true.sum()))

print("\nTarget P1 recall: >= 0.98")
print("Target achieved:", p1_recall >= 0.98)


In [ ]:

resolver_cm = confusion_matrix(
    resolver_true,
    resolver_pred,
    labels=list(range(len(resolver_labels)))
)

priority_cm = confusion_matrix(
    priority_true,
    priority_pred,
    labels=list(range(len(priority_labels)))
)

resolver_cm_df = pd.DataFrame(
    resolver_cm,
    index=resolver_labels,
    columns=resolver_labels
)

priority_cm_df = pd.DataFrame(
    priority_cm,
    index=priority_labels,
    columns=priority_labels
)

print("RESOLVER CONFUSION MATRIX")
display(resolver_cm_df)

print("\nPRIORITY CONFUSION MATRIX")
display(priority_cm_df)


In [ ]:

MODEL_OUTPUT.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(MODEL_OUTPUT))
tokenizer.save_pretrained(str(MODEL_OUTPUT))

# Save label mappings
label_mappings = {
    "resolver_labels": resolver_labels,
    "priority_labels": priority_labels,
    "resolver_mapping": resolver_mapping,
    "priority_mapping": priority_mapping
}

with open(
    MODEL_OUTPUT / "label_mappings.json",
    "w"
) as file:
    json.dump(label_mappings, file, indent=4)

trainer.save_state()

print("Model saved successfully.")
print("Location:", MODEL_OUTPUT)


In [ ]:

# Classification reports
pd.DataFrame(resolver_report).transpose().to_csv(
    REPORT_OUTPUT / "resolver_classification_report.csv"
)

pd.DataFrame(priority_report).transpose().to_csv(
    REPORT_OUTPUT / "priority_classification_report.csv"
)

# Confusion matrices
resolver_cm_df.to_csv(
    REPORT_OUTPUT / "resolver_confusion_matrix.csv"
)

priority_cm_df.to_csv(
    REPORT_OUTPUT / "priority_confusion_matrix.csv"
)

# Evaluation metrics
evaluation_summary = {
    "model": MODEL_NAME,
    "architecture": "ModernBERT multitask",
    "dataset_size": len(df),
    "training_size": len(train_df),
    "validation_size": len(val_df),
    "test_size": len(test_df),
    "resolver_classes": len(resolver_labels),
    "priority_classes": len(priority_labels),
    "p1_precision": float(p1_precision),
    "p1_recall": float(p1_recall),
    "p1_f1": float(p1_f1),
    "p1_support": int(p1_support),
    "test_metrics": {
        key: float(value)
        for key, value in test_results.items()
        if isinstance(value, (int, float, np.number))
    }
}

with open(
    REPORT_OUTPUT / "evaluation_summary.json",
    "w"
) as file:
    json.dump(evaluation_summary, file, indent=4)

# Training history
pd.DataFrame(
    trainer.state.log_history
).to_csv(
    REPORT_OUTPUT / "training_history.csv",
    index=False
)

print("Evaluation reports saved.")
print("Report location:", REPORT_OUTPUT)


In [ ]:

test_output_df = test_df[
    [
        "ticket_id",
        "text",
        "resolver_group",
        "priority"
    ]
].copy()

test_output_df["predicted_resolver_group"] = [
    resolver_labels[index]
    for index in resolver_pred
]

test_output_df["predicted_priority"] = [
    priority_labels[index]
    for index in priority_pred
]

resolver_probabilities = torch.softmax(
    torch.tensor(resolver_logits),
    dim=-1
).numpy()

priority_probabilities = torch.softmax(
    torch.tensor(priority_logits),
    dim=-1
).numpy()

test_output_df["resolver_confidence"] = (
    resolver_probabilities.max(axis=1).round(4)
)

test_output_df["priority_confidence"] = (
    priority_probabilities.max(axis=1).round(4)
)

test_output_df["resolver_correct"] = (
    test_output_df["resolver_group"] ==
    test_output_df["predicted_resolver_group"]
)

test_output_df["priority_correct"] = (
    test_output_df["priority"] ==
    test_output_df["predicted_priority"]
)

test_output_df.to_csv(
    REPORT_OUTPUT / "test_predictions.csv",
    index=False
)

print("Test predictions saved.")

print("\nResolver accuracy:")
print(round(test_output_df["resolver_correct"].mean(), 4))

print("\nPriority accuracy:")
print(round(test_output_df["priority_correct"].mean(), 4))

print("\nIncorrect resolver predictions:")
display(
    test_output_df[
        ~test_output_df["resolver_correct"]
    ].head(10)
)


In [ ]:

print("MODEL FILES")

for file in sorted(MODEL_OUTPUT.iterdir()):
    print(file.name)

print("\nREPORT FILES")

for file in sorted(REPORT_OUTPUT.iterdir()):
    print(file.name)

assert (MODEL_OUTPUT / "config.json").exists()
assert (MODEL_OUTPUT / "model.safetensors").exists()
assert (MODEL_OUTPUT / "label_mappings.json").exists()
assert (REPORT_OUTPUT / "evaluation_summary.json").exists()

print("\nAll required artifacts verified.")
